# 03 - Temporal VAE + k-means

Adds time context to the ConvVAE. Each training example is a run of 30 consecutive days. The CNN encodes each day, a one-direction LSTM reads the days in order, and the latent mean of day t comes from the day's own features concatenated with the LSTM state. Day t never sees days after t, which keeps the model usable for early warning.

Loss = reconstruction + beta * KL + lambda * smoothness, where smoothness is the squared change of the latent mean between consecutive days relative to its spread. This is the first stage of the temporal regularity idea in Nie et al. (2026).

Pipeline: 30 days of scalograms -> CNN per day -> causal LSTM -> latent mean per day + daily price level -> k-means -> regime

Methodology notes:
1. Days are split chronologically (60% train, 20% validation, 20% test). The 2023-2024 El Nino falls in the test period.
2. The k-means is fitted on training days only and then applied to all days.
3. Scalograms come from the real price with each day's mean removed, so they show only the intraday shape. `real_price_log` uses the log real price and `real_price` the real price in COP/kWh. The daily level (mean and standard deviation of the same series) is added to the latent vector before clustering.
4. Each run creates a folder in `experiments/temporal_vae/<clustering>/` with config, history, metrics, tables and figures, and adds one row to `experiments/registry.csv`.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import torch

import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split, ScalogramSequenceDataset
from src.models.temporal_vae import TemporalVAE
from src.training import set_seed, train
from src.clustering import with_level, level_columns, assign_regimes, fit_clusters, order_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import loss_curves, selection_chart, tsne_chart, regime_timeline, plot_reconstructions
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment configuration

Change only this cell between runs. `SCALOGRAM_SET` is `"real_price"` or `"real_price_log"`. `CLUSTERING` is `"kmeans"` or `"gmm"`. Start with `N_REGIMES = 2` and increase it in later runs.

`rnn_type` accepts "lstm" or "gru". `lam` is the weight of the smoothness term, which is about 1 minus the lag-1 autocorrelation of each latent dimension (0 means perfectly smooth). If the latents become almost constant, lower it; if regimes still flicker day to day, raise it. `kl_warmup_epochs` ramps the KL weight to avoid the KL spike seen in the first epoch.

In [ ]:
SCALOGRAM_SET = "real_price"
CLUSTERING = "kmeans"
N_REGIMES = 2

CONFIG = {
    "model": "temporal_vae",
    "scalogram_set": SCALOGRAM_SET,
    "clustering": CLUSTERING,
    "n_regimes": N_REGIMES,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "z_dim": 16,
    "filters": [128, 64, 32],
    "proj_dim": 128,
    "rnn_type": "lstm",
    "rnn_hidden": 64,
    "seq_len": 30,
    "stride": 7,
    "beta": 1.0,
    "lam": 100.0,
    "kl_warmup_epochs": 5,
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 16,
    "epochs": 30,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folder

In [ ]:
SCALOGRAM_DIR = Path("../data/processed/scalograms") / SCALOGRAM_SET
DATA_DIR = Path("../data/processed")

RUN_DIR = exp.create_run(CLUSTERING, SCALOGRAM_SET, CLUSTERING, N_REGIMES, root=exp.EXPERIMENTS_DIR / CONFIG["model"])
exp.save_config(RUN_DIR, CONFIG)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
print(f"Run: {exp.run_id(RUN_DIR)}")
print(f"Device: {DEVICE}")

## 2. Data

In [ ]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms ({SCALOGRAM_SET}): {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")
print(f"Daily level columns: {level_columns(metadata)}")
exp.save_table(RUN_DIR, "splits", exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx}))

In [ ]:
train_set = ScalogramSequenceDataset(images, train_idx, CONFIG["seq_len"], CONFIG["stride"])
val_set = ScalogramSequenceDataset(images, val_idx, CONFIG["seq_len"], CONFIG["stride"])
train_loader = DataLoader(train_set, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG["batch_size"], shuffle=False)
print(f"Sequences of {CONFIG['seq_len']} days: train {len(train_set)}, val {len(val_set)}")

## 3. Model

In [ ]:
model = TemporalVAE(CONFIG["z_dim"], IMG_C, IMG_H, IMG_W, CONFIG["filters"],
                    CONFIG["proj_dim"], CONFIG["rnn_hidden"], CONFIG["rnn_type"]).to(DEVICE)

with torch.no_grad():
    sample = train_set[0][0].unsqueeze(0).to(DEVICE)
    out_shape = tuple(model(sample)[0].shape[1:])
assert out_shape == tuple(sample.shape[1:]), f"decoder output {out_shape} does not match input"

N_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Encoder feature map: {model.encoder.feat_shape}")
print(f"Input sequence: {tuple(sample.shape[1:])}")
exp.save_table(RUN_DIR, "hyperparameters", exp.hyperparameter_table(CONFIG, N_PARAMS))

## 4. Training

In [ ]:
history = train(model, train_loader, val_loader, DEVICE, epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                beta=CONFIG["beta"], kl_warmup_epochs=CONFIG["kl_warmup_epochs"],
                extra_loss_kwargs={"lam": CONFIG["lam"]})
exp.save_history(RUN_DIR, history)
exp.save_model(RUN_DIR, model)

## 5. Loss curves

Posterior collapse check: if the KL curve drops close to zero, the encoder is ignoring the input. In that case try `beta` below 1 or set `kl_warmup_epochs`. Active dimensions counts the latent dimensions whose mean actually varies across days.

In [ ]:
loss_plot = loss_curves(history, components=("total", "recon", "kl", "smooth"))
loss_plot

In [ ]:
loss_plot.save(exp.figure_path(RUN_DIR, "loss_curves"), scale_factor=2.0)

## 6. Reconstructions

Eight consecutive days from the test period.

In [ ]:
model.eval()
start = test_idx[len(test_idx) // 2]
seq = torch.from_numpy(images[start:start + CONFIG["seq_len"]]).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    recon = model(seq)[0][0].cpu().numpy()

show = np.arange(CONFIG["seq_len"] - 8, CONFIG["seq_len"])
fig = plot_reconstructions(images[start + show], recon[show], [str(d.date()) for d in context["date"].iloc[start + show]])
fig.savefig(exp.figure_path(RUN_DIR, "reconstructions"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent vectors

Each day's vector is computed from that day and the 29 days before it, the same context the model saw in training.

In [ ]:
latents = model.extract_latents(images, DEVICE, seq_len=CONFIG["seq_len"])
exp.save_latents(RUN_DIR, latents)
features = with_level(latents, metadata, train_idx)

active = int((latents.var(axis=0) > 0.01).sum())
print(f"Latent vectors: {latents.shape}")
print(f"Final train KL: {history[-1]['train_kl']:.2f}")
print(f"Active dimensions: {active} of {CONFIG['z_dim']}")
print(f"Clustering features (latents + daily level): {features.shape}")

## 8. Regimes with k-means

K-means uses the latent vector plus the daily price level (mean and standard deviation, z-scored on training days), the same input as the GMM notebooks, because the scalograms only carry the intraday shape. It is fitted on the training days only and then assigns every day. k comes from `N_REGIMES`. The table shows, for each k on the training days, the inertia (look for the elbow, the role the BIC plays for the GMM), silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better). Regimes are numbered by ascending mean real price, so regime 0 is always the cheapest and labels are comparable across models.

In [ ]:
selection, K, labels, probs = assign_regimes(
    features, train_idx, context["precio_real_cop_kwh"].to_numpy(),
    k=N_REGIMES, k_range=CONFIG["k_range"], seed=CONFIG["seed"], method=CLUSTERING,
)
context["regime"] = labels

exp.save_table(RUN_DIR, "k_selection", exp.selection_table(selection))
print(f"Selected k = {K}")

In [ ]:
selection_plot = selection_chart(selection, K)
selection_plot

In [ ]:
selection_plot.save(exp.figure_path(RUN_DIR, "k_selection"), scale_factor=2.0)

## 9. Latent space

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=CONFIG["seed"]).fit_transform(features[plot_idx])

tsne_plot = tsne_chart(coords, labels[plot_idx], f"t-SNE of the latent space (k={K})")
tsne_plot

In [ ]:
tsne_plot.save(exp.figure_path(RUN_DIR, "tsne_regimes"), scale_factor=2.0)

## 10. Regime timeline

Shaded bands are El Nino months (ONI >= 0.5). Dashed lines mark the start of the validation and test periods. First sanity check: the high-price regimes should concentrate in the 2015-2016 and 2023-2024 El Nino episodes and in low reservoir periods.

In [ ]:
timeline_plot = regime_timeline(context, split_dates, title=f"Detected regimes over time (k={K})")
timeline_plot

In [ ]:
timeline_plot.save(exp.figure_path(RUN_DIR, "regime_timeline"), scale_factor=2.0)

## 11. Regime profile and clustering metrics

Clustering metrics are computed on all days with the clustering features (latents + daily level). The last cell writes `regimes.csv` and `metrics.json` and adds the run to `experiments/registry.csv`.

In [ ]:
exp.save_table(RUN_DIR, "regime_profile", exp.regime_profile_table(regime_profile(context)))

In [ ]:
metrics = {"run_id": exp.run_id(RUN_DIR), "model": CONFIG["model"], "scalogram_set": SCALOGRAM_SET,
           "clustering": CLUSTERING, "k": K, **summary_metrics(features, labels, seed=CONFIG["seed"])}
save_regimes(RUN_DIR, context, labels, probs, metrics)
exp.save_table(RUN_DIR, "clustering_metrics", exp.metrics_table(metrics))
exp.log_run(RUN_DIR, CONFIG, metrics, registry=exp.EXPERIMENTS_DIR / "registry.csv")

In [ ]:
spread = context.groupby("regime")["precio_real_cop_kwh"].describe(percentiles=[0.1, 0.5, 0.9])
spread[["count", "10%", "50%", "90%", "min", "max"]].round(1)

## 12. Exploratory checks

Not saved to the run. Compares clustering inputs with the same algorithm and k as this run.

In [ ]:
print("Latent variance per dim:", np.round(latents.var(axis=0), 5))
print("Latent std range:", latents.std(axis=0).min().round(4), "to", latents.std(axis=0).max().round(4))

In [ ]:
price = context["precio_real_cop_kwh"].to_numpy()
lvl_cols = list(level_columns(metadata))
meta7 = metadata.copy()
meta7[lvl_cols] = metadata[lvl_cols].rolling(7, min_periods=1).mean()
empty = np.zeros((len(latents), 0))
strong = np.argsort(latents.var(axis=0))[-3:]

rows = (
    ("level only", with_level(empty, metadata, train_idx)),
    ("level 7d", with_level(empty, meta7, train_idx)),
    ("latents only", latents),
    ("3 strongest latents", latents[:, strong]),
    ("latents + level", features),
    ("latents + level 7d", with_level(latents, meta7, train_idx)),
    ("3 strongest + level 7d", with_level(latents[:, strong], meta7, train_idx)),
)
t = PrettyTable(["Features", "Silhouette", "Davies-Bouldin", "Mean run", "Switches / yr", "Median price by regime"])
for name, f in rows:
    lab = order_regimes(fit_clusters(f, train_idx, K, CLUSTERING, CONFIG["seed"])[0], price)
    m = summary_metrics(f, lab, seed=CONFIG["seed"])
    med = context.assign(r=lab).groupby("r")["precio_real_cop_kwh"].median().round(0).tolist()
    t.add_row([name, f"{m['silhouette']:.3f}", f"{m['davies_bouldin']:.3f}", f"{m['mean_run_days']:.1f}",
               f"{m['switches_per_year']:.0f}", med])
print(t)
ac = [np.corrcoef(latents[1:, j], latents[:-1, j])[0, 1] for j in range(latents.shape[1])]
print("Lag-1 autocorrelation per dim:", np.round(ac, 2))
print("Strongest dims:", strong)